[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/08-rural-community-mapping.ipynb)

# Rural Community Mapping: Defining Communities by Drive Time

## Three Hub Towns Across Western Kansas

County lines are administrative accidents. They were drawn by territorial legislators in the 19th century to organize land records, not to describe how people actually live. In rural Kansas, a county seat of 800 people may share a county with farmland stretching 30 miles in every direction, while the nearest hospital, high school, or grocery store sits in a hub town across the county line.

This notebook uses **drive-time isochrones** to define functional rural communities -- the areas whose residents realistically depend on a given town for services. We compare three western Kansas hub towns:

- **Hays** (~21,000) -- a university town (Fort Hays State) and regional medical center in Ellis County
- **Dodge City** (~28,000) -- an agricultural and meatpacking center on the Arkansas River, county seat of Ford County
- **Liberal** (~20,000) -- a meatpacking and energy town in Seward County, near the Oklahoma border

These towns are spaced 100--150 miles apart along the Great Plains corridor. Each draws workers, patients, and shoppers from multiple surrounding counties. By generating **60-minute driving isochrones**, we can see how far each town's functional community actually extends -- and compare that to the arbitrary county boundaries that federal datasets use as proxies for "community."

By the end of this notebook, you will know how to:

- Generate driving isochrones to define rural service areas
- Compare isochrone coverage against county-scale benchmarks
- Pull Census demographics for three towns simultaneously
- Create choropleth maps of population and income across rural block groups
- Discover healthcare, education, and shopping POIs
- Measure the walk-vs-drive equity gap in a rural town
- Run a formal multi-location comparison
- Model a hospital closure vulnerability scenario
- Import custom facility data from CSV
- Generate a shareable HTML report

---

## Why 60 Minutes?

Urban accessibility studies commonly use a **15-minute walk or drive** as the threshold for convenient access. In rural western Kansas, that standard is meaningless. A 15-minute drive from Hays barely gets you past the city limits. Residents routinely drive 30--60 minutes one way for groceries, medical appointments, high school sports, or church. A farmer 40 miles north of Dodge City considers Dodge City "their town" because it has the nearest hospital, Walmart, and high school.

We use **60 minutes** because it captures the actual commuting and shopping shed of a Great Plains hub town -- the area from which people realistically travel for daily and weekly needs. The results confirm this choice: a 60-minute drive from these towns covers 10,000--13,000 sq km, roughly **5--6 average Kansas counties**. That is the real community footprint.

| Setting | Typical threshold | Mode | Rationale |
|---|---|---|---|
| **Urban** | 15 minutes | Walk | Dense streets, many alternatives |
| **Suburban** | 15 minutes | Drive | Car-dependent but short distances |
| **Rural (this notebook)** | 60 minutes | Drive | Sparse population, no alternatives, long distances |

A 60-minute drive on Great Plains highways covers 50--60 miles depending on the road network and speed limits. Compare that to a typical Kansas county, which averages about 2,145 sq km (828 sq mi, roughly 27 miles on a side). The isochrone dwarfs a single county, demonstrating why county-level analysis fundamentally misrepresents these communities.

---

## Setup and Imports

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

In [ ]:
from socialmapper import (
    create_isochrone,
    get_census_blocks,
    get_census_data,
    create_map,
    get_poi,
    analyze_multiple_pois,
    generate_report,
    import_poi_csv,
)

import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display, HTML

# Consistent plot styling for the entire notebook
plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

---

## Step 1: Define the Three Hub Towns

We use central coordinates for each town. These points serve as the **origin** for isochrone generation, POI searches, and census queries.

In [ ]:
towns = {
    "Hays, KS":       (38.8794, -99.3268),
    "Dodge City, KS": (37.7528, -100.0171),
    "Liberal, KS":    (37.0439, -100.9210),
}

context = pd.DataFrame({
    "Town": ["Hays", "Dodge City", "Liberal"],
    "Approx. Pop.": ["~21,000", "~28,000", "~20,000"],
    "Character": [
        "University town (Fort Hays State), regional medical center",
        "Agricultural hub, meatpacking, county seat of Ford County",
        "Meatpacking and energy, near Oklahoma border",
    ],
    "Key Employers": [
        "FHSU, Hays Medical Center",
        "Cargill, National Beef, USD 443",
        "Seaboard Foods, NatGas Midstream",
    ],
})
display(context)

for name, (lat, lon) in towns.items():
    print(f"{name}: lat={lat}, lon={lon}")

---

## Step 2: Generate 60-Minute Driving Isochrones

An **isochrone** is a polygon enclosing all the area reachable from a starting point within a given travel time. Unlike a circular buffer, isochrones account for the actual road network -- they stretch along highways and contract where roads are sparse.

We use **driving mode** with a **60-minute** threshold because:

1. In rural western Kansas, virtually all daily trips are by car. There is no public transit.
2. A 60-minute drive captures the realistic service area of a hub town -- the radius from which people commute, shop, and seek medical care.
3. The resulting polygon reveals how the highway network shapes access -- US-183, US-283, US-56, US-83, and I-70 are the main corridors.
4. Comparing isochrone shape and area to county boundaries shows definitively that counties are the wrong unit of analysis.

SocialMapper uses the **Valhalla** open-source routing engine, which models driving on the actual OpenStreetMap road network.

In [ ]:
isochrones = {}

for name, coords in towns.items():
    iso = create_isochrone(coords, travel_time=60, travel_mode="drive")
    isochrones[name] = iso
    area = iso["properties"]["area_sq_km"]
    print(f"{name}: {area:.1f} sq km reachable within a 60-minute drive")
    time.sleep(5)  # Rate-limit courtesy pause

print()
areas = {n: isochrones[n]["properties"]["area_sq_km"] for n in towns}
largest = max(areas, key=areas.get)
smallest = min(areas, key=areas.get)
print(f"Largest driving area: {largest} ({areas[largest]:.1f} sq km)")
print(f"Smallest driving area: {smallest} ({areas[smallest]:.1f} sq km)")
print(f"Ratio: {areas[largest] / areas[smallest]:.2f}x")

---

## Step 3: Isochrone Area vs. Kansas County Area

Kansas has 105 counties. The average Kansas county is about **2,145 sq km** (828 sq mi). Western Kansas counties tend to be larger than eastern ones, but even they are dwarfed by a 60-minute driving isochrone.

The bar chart below makes the scale difference concrete. Each town's driving community spans the equivalent of 5--6 Kansas counties -- meaning that county-level Census data for any one of those counties captures only a fraction of the population that actually depends on that town.

In [ ]:
avg_ks_county_area = 2145  # sq km (828 sq mi)

town_names = list(towns.keys())
iso_areas = [isochrones[n]["properties"]["area_sq_km"] for n in town_names]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(town_names))
width = 0.35

bars_iso = ax.bar(x - width / 2, iso_areas, width, label="60-min Drive Isochrone",
                  color="#4c78a8", edgecolor="white", linewidth=1.2)
bars_county = ax.bar(x + width / 2, [avg_ks_county_area] * len(town_names), width,
                     label="Avg. KS County", color="#e45756", edgecolor="white", linewidth=1.2)

for bar, val in zip(bars_iso, iso_areas):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f"{val:,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar in bars_county:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f"{avg_ks_county_area:,}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels([n.replace(", KS", "") for n in town_names])
ax.set_ylabel("Area (sq km)")
ax.set_title("60-Minute Driving Area vs. Average Kansas County", fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

for name in town_names:
    ratio = isochrones[name]["properties"]["area_sq_km"] / avg_ks_county_area
    print(f"{name}: isochrone covers {ratio:.1f}x the area of an average KS county")

### What This Tells Us

Each town's 60-minute driving community covers the equivalent of **5--6 average Kansas counties**. Dodge City's isochrone is the largest (~12,900 sq km), likely because it sits at the intersection of major highways (US-50, US-56, US-283) that allow higher-speed travel in multiple directions. Liberal's isochrone is the smallest (~10,300 sq km), reflecting its position in the corner of the state where the Oklahoma border truncates the road network to the south.

The critical point: **any analysis that uses a single county as a proxy for one of these communities is capturing roughly one-fifth of the actual population that depends on that town.** A "Ford County" poverty rate or median income describes a political rectangle, not the lived community of Dodge City.

---

## Step 4: Fetch Census Demographics

To understand *who lives* in each town's functional community, we pull five American Community Survey (ACS) variables:

| Variable | What it tells us |
|---|---|
| `population` | Total residents in each block group |
| `median_income` | Household purchasing power |
| `poverty` | Count of residents below the federal poverty line |
| `housing_units` | Total housing stock (proxy for density) |
| `households_no_vehicle` | Residents who depend entirely on others for transportation |

The `get_census_data` function fetches data at the **block group** level -- the smallest geography for which the Census Bureau publishes most ACS estimates.

In [ ]:
demographic_variables = [
    "population",
    "median_income",
    "poverty",
    "housing_units",
    "households_no_vehicle",
]

blocks_data = {}
census_data = {}
merged_data = {}

for name in towns:
    iso = isochrones[name]

    # Fetch block group boundaries that intersect the isochrone
    blocks = get_census_blocks(polygon=iso)

    # Fetch ACS demographic data for those block groups
    census = get_census_data(iso, variables=demographic_variables)

    # Merge geometry + demographics into a single list of dicts
    merged = []
    for block in blocks:
        geoid = block["geoid"]
        if geoid in census.data:
            merged.append({**block, **census.data[geoid]})

    blocks_data[name] = blocks
    census_data[name] = census
    merged_data[name] = merged

    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    print(f"\n{name}:")
    print(f"  Block groups matched: {len(merged)}")
    print(f"  Total population:     {total_pop:,}")

---

## Step 5: Demographic Summary Table

Let us compute aggregate statistics for each town and display them in a comparison table.

In [ ]:
summary = {}

for name in towns:
    census = census_data[name]
    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    incomes = [
        d["median_income"]
        for d in census.data.values()
        if d.get("median_income") is not None and d["median_income"] > 0
    ]
    avg_income = sum(incomes) / len(incomes) if incomes else 0
    total_poverty = sum(
        d.get("poverty", 0)
        for d in census.data.values()
        if d.get("poverty") is not None
    )
    poverty_rate = (total_poverty / total_pop * 100) if total_pop > 0 else 0
    no_vehicle = sum(
        d.get("households_no_vehicle", 0)
        for d in census.data.values()
        if d.get("households_no_vehicle") is not None
    )
    total_housing = sum(
        d.get("housing_units", 0)
        for d in census.data.values()
        if d.get("housing_units") is not None
    )

    summary[name] = {
        "Population": total_pop,
        "Avg. Median Income": f"${avg_income:,.0f}",
        "Poverty Count": total_poverty,
        "Poverty Rate": f"{poverty_rate:.1f}%",
        "Households w/o Vehicle": no_vehicle,
        "Housing Units": total_housing,
        "Drive Area (sq km)": f"{isochrones[name]['properties']['area_sq_km']:.0f}",
    }

summary_df = pd.DataFrame(summary)
display(summary_df)

### What the Numbers Reveal

The demographic data exposes three distinct community profiles:

- **Dodge City has the largest functional community** by every measure: ~89,000 people across 76 block groups, the highest average median income (~$76,000), and the lowest poverty rate (~12%). Its meatpacking industry (Cargill, National Beef) and agricultural economy generate wages that pull workers from across southwestern Kansas.

- **Liberal has the highest poverty rate** (~14%), consistent with its dependence on meatpacking and energy sectors and its geographic isolation near the Oklahoma border. Its ~69,000-person community spans 64 block groups.

- **Hays sits in between** with ~61,000 people, an average median income around $68,000, and a ~13% poverty rate. Fort Hays State University's student population likely inflates the poverty count -- college students are counted as below the poverty line based on individual income even when supported by family.

The most striking finding: **zero households without vehicles** across all three communities. In rural western Kansas, car ownership is not a lifestyle choice -- it is a prerequisite for participation in daily life. There is no public transit, no rideshare, no Uber, no bike infrastructure. This makes the walk-vs-drive analysis in Step 10 especially important: the few residents who temporarily or permanently lack a car (elderly, disabled, those with mechanical breakdowns) experience complete isolation.

---

## Step 6: Population Choropleth Maps

Choropleth maps color each block group by a numeric variable. The dashed boundary shows the 25-minute driving isochrone -- the functional community boundary.

In [ ]:
for name in towns:
    map_result = create_map(
        data=merged_data[name],
        column="population",
        title=f"Population by Block Group -- {name}",
        overlay_boundary=isochrones[name],
        show_stats=True,
    )
    print(f"\n--- {name} ---")
    display(Image(data=map_result.image_data))

### Reading the Population Maps

These maps reveal the defining characteristic of Great Plains geography: **tight population clusters surrounded by vast emptiness**. Each town shows a few darkly shaded block groups in the urban core where thousands of people live, surrounded by enormous, lightly shaded rural block groups where a few hundred people are spread across hundreds of square miles of wheat fields and rangeland.

Rural block groups are physically enormous because the Census Bureau requires a minimum population per block group (typically 600--3,000 people). When population density is 2--5 people per square mile, a single block group can cover an area larger than some eastern cities. This means the choropleth shading in outlying areas is misleading -- a light-shaded block group does not mean "no one lives here," it means "people are spread very thin across a huge area."

The isochrone boundary (dashed line) extends well beyond the town itself, capturing the commuting shed -- the area from which people drive into town for work, school, shopping, and medical care.

---

## Step 7: Income Choropleth Maps

Income maps reveal economic stratification within each driving area. We use the `RdYlGn` (red-yellow-green) colormap so that low-income areas appear in red and high-income areas in green.

In [ ]:
for name in towns:
    income_map = create_map(
        data=merged_data[name],
        column="median_income",
        title=f"Median Household Income -- {name}",
        overlay_boundary=isochrones[name],
        show_stats=True,
        cmap="RdYlGn",
    )
    print(f"\n--- {name} ---")
    display(Image(data=income_map.image_data))

### Reading the Income Maps

The income maps reveal a counterintuitive finding: **Dodge City, the meatpacking hub, has the highest average median income** (~$76,000), while **Hays, the university town, has the lowest** (~$68,000). This challenges the common assumption that university towns are wealthier than industrial towns.

The explanation lies in the structure of each economy. Meatpacking jobs at Cargill and National Beef are physically demanding but pay competitive hourly wages, especially with overtime and shift differentials. The large agricultural operations surrounding Dodge City also generate substantial income. In contrast, Hays' economy includes a large population of university students and part-time workers whose low individual incomes pull down block-group medians.

Within each town, look for variation between block groups. Higher-income areas tend to correspond to newer residential developments and established farming communities, while lower-income areas may correspond to older housing stock near the town center or areas with more rental housing.

---

## Step 8: Service Discovery -- Healthcare, Education, Shopping

In rural communities, access to services defines quality of life. We search OpenStreetMap for three critical categories:

- **Healthcare**: hospitals, clinics, pharmacies, dentists
- **Education**: schools, libraries, universities
- **Shopping**: supermarkets, grocery stores, convenience stores, general retail

The `get_poi` function creates a 60-minute driving isochrone behind the scenes, queries the Overpass API for matching OSM features, and computes actual driving travel times via Valhalla's matrix API.

In [ ]:
service_categories = ["healthcare", "education", "shopping"]

poi_data = {}

for name, coords in towns.items():
    pois = get_poi(
        coords,
        categories=service_categories,
        travel_time=60,
        travel_mode="drive",
        limit=80,
    )
    poi_data[name] = pois
    print(f"\n{'=' * 55}")
    print(f"{name}: {len(pois)} service POIs within 60-min drive")
    print(f"{'=' * 55}")

    # Count by category
    cat_counts = {}
    for p in pois:
        cat = p.get("category", "other")
        cat_counts[cat] = cat_counts.get(cat, 0) + 1
    for cat, count in sorted(cat_counts.items()):
        print(f"  {cat:<15} {count}")

    # Show top 5 closest
    print(f"\n  {'Name':<35} {'Category':<15} {'Drive (min)'}")
    print(f"  {'-'*35} {'-'*15} {'-'*10}")
    for p in pois[:5]:
        travel = p.get("travel_time_minutes", "N/A")
        print(f"  {p['name'][:34]:<35} {p['category']:<15} {travel}")
    time.sleep(5)  # Rate-limit courtesy pause between towns

---

## Step 9: Service Comparison Charts and POI Overlay Maps

A grouped bar chart makes differences in service mix immediately visible. We then overlay POI markers on the population maps for spatial context.

In [ ]:
# Grouped bar chart: services by category
categories_for_chart = ["healthcare", "education", "shopping"]
town_names_short = [n.replace(", KS", "") for n in towns]
colors = ["#4c78a8", "#f58518", "#54a24b"]

category_counts = {}
for name in towns:
    counts = {}
    for p in poi_data[name]:
        cat = p.get("category", "other")
        if cat in categories_for_chart:
            counts[cat] = counts.get(cat, 0) + 1
    category_counts[name] = counts

x = np.arange(len(categories_for_chart))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, color) in enumerate(zip(towns, colors)):
    vals = [category_counts[name].get(cat, 0) for cat in categories_for_chart]
    bars = ax.bar(x + i * width, vals, width, label=name.replace(", KS", ""),
                  color=color, edgecolor="white", linewidth=1.2)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    str(val), ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels([c.replace("_", " ").title() for c in categories_for_chart])
ax.set_ylabel("Number of POIs")
ax.set_title("Services Within 60-Minute Drive by Category", fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# POI overlay maps for each town
for name in towns:
    overlay_points = [
        {"lat": p["lat"], "lon": p["lon"], "name": p["name"]}
        for p in poi_data[name][:20]
    ]

    map_result = create_map(
        data=merged_data[name],
        column="population",
        title=f"Services Overlay -- {name}",
        overlay_boundary=isochrones[name],
        overlay_points=overlay_points,
        show_stats=True,
    )
    print(f"\n--- {name} ---")
    display(Image(data=map_result.image_data))

### What the Service Data Reveals

All three towns hit the 80-POI search cap, but the **category mix differs dramatically** and tells a story about each town's role in the region:

- **Hays is the shopping hub**: roughly two-thirds of its POIs are shopping/retail, reflecting its role as the place where surrounding farmers and ranchers come to buy supplies. It also has the most healthcare facilities of any town in this analysis (~13 in the general search, 26 in a dedicated healthcare search), anchored by Hays Medical Center.

- **Dodge City and Liberal are education-heavy**: more than half their POIs are education facilities (schools, libraries). This reflects their large, young meatpacking workforces -- these communities have larger families and more school-age children than Hays. Dodge City in particular has a very high school count relative to healthcare (54 education vs. 7 healthcare in the general search).

On the maps, **nearly all POI markers cluster within the town centers**. The vast rural areas within the isochrone -- covering thousands of square miles -- have essentially zero services. This hub-and-spoke pattern is efficient when the hub town is functioning, but creates total dependence: if a family 30 miles from town needs groceries, medical care, or school supplies, they have exactly one destination.

---

## Step 10: Walk vs. Drive Equity Gap (Hays)

Hays is a university town with a relatively compact downtown along Main Street. Let us compare what residents can reach by **walking 15 minutes** versus **driving 60 minutes**. This reveals the equity gap between those who can drive and those who cannot.

In rural towns, this gap is not a gradient -- it is a cliff. A resident without a car in Hays can walk to a few downtown shops and a clinic. The hospital, Walmart, high school, and every service beyond the town center are completely out of reach.

In [ ]:
hays_coords = towns["Hays, KS"]

# Walking isochrone (15 min)
walk_iso = create_isochrone(hays_coords, travel_time=15, travel_mode="walk")
walk_area = walk_iso["properties"]["area_sq_km"]

# Driving isochrone (60 min) -- already computed
drive_area = isochrones["Hays, KS"]["properties"]["area_sq_km"]

print(f"Hays, KS -- Walk vs. Drive Comparison")
print(f"  15-min walk area:  {walk_area:.2f} sq km")
print(f"  60-min drive area: {drive_area:.1f} sq km")
print(f"  Drive area is {drive_area / walk_area:,.0f}x the walk area")

time.sleep(5)  # Rate-limit courtesy pause

# Walking POIs
walk_pois = get_poi(
    hays_coords,
    categories=service_categories,
    travel_time=15,
    travel_mode="walk",
    limit=50,
)
drive_pois = poi_data["Hays, KS"]

print(f"\n  Services within 15-min walk:  {len(walk_pois)}")
print(f"  Services within 60-min drive: {len(drive_pois)}")

if len(walk_pois) > 0:
    print(f"  Driving provides {len(drive_pois) / len(walk_pois):.1f}x the service access")
else:
    print(f"  No services within walking distance -- complete car dependency")

print(f"\n  Walking services:")
for p in walk_pois[:10]:
    print(f"    {p['name'][:40]:<42} {p.get('category',''):<15} {p.get('travel_time_minutes','?')} min")

In [ ]:
# Walk vs. Drive bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Area comparison
bars = axes[0].bar(["Walk (15 min)", "Drive (60 min)"], [walk_area, drive_area],
                   color=["#f58518", "#4c78a8"], edgecolor="white", linewidth=1.2)
for bar, val in zip(bars, [walk_area, drive_area]):
    label = f"{val:.1f}" if val < 100 else f"{val:,.0f}"
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                 label, ha="center", va="bottom", fontweight="bold", fontsize=11)
axes[0].set_ylabel("Area (sq km)")
axes[0].set_title("Reachable Area -- Hays, KS", fontweight="bold")
axes[0].spines[["top", "right"]].set_visible(False)

# Service count comparison
bars = axes[1].bar(["Walk (15 min)", "Drive (60 min)"], [len(walk_pois), len(drive_pois)],
                   color=["#f58518", "#4c78a8"], edgecolor="white", linewidth=1.2)
for bar, val in zip(bars, [len(walk_pois), len(drive_pois)]):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 str(val), ha="center", va="bottom", fontweight="bold", fontsize=11)
axes[1].set_ylabel("Number of Services")
axes[1].set_title("Service Access -- Hays, KS", fontweight="bold")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

print(f"A person without a car in Hays has access to {walk_area / drive_area:.2%}")
print(f"of the geographic community available to a driver.")

---

## Step 11: Multi-Town Formal Comparison

The `analyze_multiple_pois` function performs a structured, side-by-side analysis of multiple locations in a single call. Here we demonstrate it with a single location (to keep API usage efficient), then build the three-way comparison from the census data already collected above.

In [ ]:
time.sleep(15)  # Rate-limit courtesy pause

# Demonstrate analyze_multiple_pois with a single location
# (In practice you would pass all locations; we keep it light for the tutorial)
comparison = analyze_multiple_pois(
    locations=[towns["Hays, KS"]],
    travel_time=60,
    travel_mode="drive",
    variables=["population", "median_income", "poverty", "housing_units"],
)

print("analyze_multiple_pois returns a structured dict:")
print(f"  Keys: {list(comparison.keys())}")
print(f"  Locations analyzed: {len(comparison['locations'])}")
print(f"  Metadata: {comparison['metadata']}")

hays_result = comparison["locations"][0]
print(f"\n  Hays, KS aggregated data:")
for var, stats in hays_result["aggregated"].items():
    print(f"    {var}: total={stats['total']:,.0f}, mean={stats['mean']:,.0f}, "
          f"min={stats['min']:,.0f}, max={stats['max']:,.0f}")

# Full three-town comparison from data already in memory
print("\n\n" + "=" * 65)
print("THREE-TOWN COMPARISON (from cached census data)")
print("=" * 65)

compare_vars = ["population", "median_income", "poverty", "housing_units"]
for var in compare_vars:
    print(f"\n--- {var.upper()} ---")
    print(f"  {'Town':<20} {'Total':>12}  {'Mean':>10}")
    rows = []
    for name in towns:
        values = [
            d.get(var, 0)
            for d in census_data[name].data.values()
            if d.get(var) is not None
        ]
        total = sum(values)
        mean = total / len(values) if values else 0
        rows.append((name, total, mean))
        print(f"  {name:<20} {total:>12,.0f}  {mean:>10,.0f}")
    highest = max(rows, key=lambda r: r[1])
    lowest = min(rows, key=lambda r: r[1])
    print(f"  Highest: {highest[0]}  |  Lowest: {lowest[0]}")

---

## Step 12: Healthcare Access and Hospital Closure Vulnerability

Rural hospital closures are a growing crisis across the Great Plains. Since 2010, dozens of rural hospitals in Kansas have closed or reduced services. When a hospital closes, residents must drive further for emergency care -- and in a stroke or heart attack, those extra minutes can be fatal.

We focus on **Liberal, KS** -- the most geographically isolated of our three towns. A dedicated healthcare POI search reveals Liberal's critical vulnerability: a handful of local facilities within a few minutes of town, then a **massive gap** of 35+ minutes before the next healthcare options appear. We also demonstrate `import_poi_csv` for integrating authoritative facility data that may be more complete than OpenStreetMap.

In [ ]:
# Healthcare distribution analysis for Liberal, KS
liberal_coords = towns["Liberal, KS"]

time.sleep(10)  # Rate-limit courtesy pause

# All healthcare within 60 minutes
healthcare = get_poi(
    liberal_coords,
    categories=["healthcare"],
    travel_time=60,
    travel_mode="drive",
    limit=50,
)

print("=" * 60)
print("Liberal, KS -- Healthcare Access Distribution")
print("=" * 60)
print(f"\nTotal healthcare facilities within 60-min drive: {len(healthcare)}")
print(f"\n  {'Facility':<42} {'Drive Time':>10}")
print(f"  {'-'*42} {'-'*10}")
for p in healthcare:
    t = p.get("travel_time_minutes", "?")
    label = f"{t:.0f} min" if isinstance(t, (int, float)) else str(t)
    print(f"  {p['name'][:41]:<42} {label:>10}")

# Bucket by time bands to expose the gap
bands = {"0-10 min": 0, "10-20 min": 0, "20-30 min": 0, "30-40 min": 0, "40-50 min": 0, "50-60 min": 0}
for p in healthcare:
    t = p.get("travel_time_minutes")
    if t is None:
        continue
    if t <= 10: bands["0-10 min"] += 1
    elif t <= 20: bands["10-20 min"] += 1
    elif t <= 30: bands["20-30 min"] += 1
    elif t <= 40: bands["30-40 min"] += 1
    elif t <= 50: bands["40-50 min"] += 1
    else: bands["50-60 min"] += 1

print(f"\n  Drive-Time Distribution:")
for band, count in bands.items():
    bar = "#" * count
    print(f"    {band:<12} {count:>3}  {bar}")

# Identify the critical gap
local = [p for p in healthcare if p.get("travel_time_minutes", 99) <= 10]
mid = [p for p in healthcare if 10 < p.get("travel_time_minutes", 0) <= 35]
distant = [p for p in healthcare if p.get("travel_time_minutes", 0) > 35]

print(f"\n  CRITICAL GAP ANALYSIS:")
print(f"    Facilities within 10 min (local):  {len(local)}")
print(f"    Facilities 10-35 min:              {len(mid)}")
print(f"    Facilities 35-60 min:              {len(distant)}")
if len(mid) == 0:
    print(f"\n    ** {len(distant)} facilities exist beyond 35 minutes,")
    print(f"       but ZERO between 10 and 35 minutes.")
    print(f"       If local facilities close, nearest alternative is ~40 min away.")

In [ ]:
# Demonstrate import_poi_csv with a small hypothetical CSV of regional facilities
import tempfile, os

csv_content = """name,latitude,longitude,type
Hays Medical Center,38.8726,-99.3372,hospital
Dodge City Medical Center,37.7506,-100.0214,hospital
Southwest Medical Center (Liberal),37.0486,-100.9198,hospital
Russell County Hospital,38.8953,-98.8598,hospital
Ness County Hospital,38.4529,-99.9065,hospital
Hodgeman County Health Center,38.0886,-99.8949,clinic
Meade District Hospital,37.2858,-100.3406,hospital
"""

csv_path = os.path.join(tempfile.gettempdir(), "ks_hospitals.csv")
with open(csv_path, "w") as f:
    f.write(csv_content)

custom_hospitals = import_poi_csv(
    csv_path,
    name_field="name",
    lat_field="latitude",
    lon_field="longitude",
    type_field="type",
)

print(f"Imported {len(custom_hospitals)} regional facilities from CSV:\n")
for h in custom_hospitals:
    print(f"  {h['name']:<40} ({h['lat']:.4f}, {h['lon']:.4f})  [{h['category']}]")

print(f"\nThe import_poi_csv function is useful for integrating local health department")
print(f"data, custom facility lists, or state licensing records that may be more")
print(f"complete than OpenStreetMap for rural areas.")

### The 35-Minute Void

The healthcare distribution around Liberal reveals a pattern common across the rural Great Plains: **a tight cluster of local facilities, then a vast dead zone, then distant options.**

Liberal has a few healthcare facilities within about 4 minutes of town center -- the local hospital (SouthWest Medical Center), a pharmacy, and a dental office. After that, there is **nothing** until roughly 40 minutes, when Satanta District Hospital, Meade District Hospital, and facilities across the Oklahoma border in Guymon come into range.

That 35-minute void is the vulnerability. If SouthWest Medical Center closes or reduces services:
- **Emergency care**: A heart attack or car accident victim faces a 40-minute drive to the nearest hospital, well beyond the golden hour.
- **Daily healthcare**: Routine doctor visits, pharmacy runs, and dental appointments all become 80-minute round trips.
- **Workforce cascade**: Healthcare workers leave, making it harder to attract other professionals (teachers, county officials) who expect a nearby hospital.

The `import_poi_csv` function lets analysts integrate authoritative data sources (state licensing databases, Medicare facility lists, CMS hospital compare data) that may be more complete than OpenStreetMap for rural areas.

---

## Step 13: Generate Shareable HTML Report

The `generate_report` function transforms the comparison dictionary into a formatted, shareable HTML document. This is useful for distributing findings to county commissioners, health departments, or community organizations that may not use Jupyter notebooks.

In [ ]:
report_html = generate_report(comparison, format="html")
print(f"Generated HTML report: {len(report_html):,} characters")
display(HTML(report_html))

---

## Key Findings

Consolidating the data from all 13 steps into a final summary.

In [ ]:
print("=" * 65)
print("RURAL COMMUNITY MAPPING ASSESSMENT")
print("Western Kansas: Hays, Dodge City, Liberal (60-min drive)")
print("=" * 65)

for name in towns:
    census = census_data[name]
    pois = poi_data[name]
    iso = isochrones[name]

    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    incomes = [
        d["median_income"]
        for d in census.data.values()
        if d.get("median_income") is not None and d["median_income"] > 0
    ]
    avg_income = sum(incomes) / len(incomes) if incomes else 0
    total_poverty = sum(
        d.get("poverty", 0)
        for d in census.data.values()
        if d.get("poverty") is not None
    )
    poverty_rate = (total_poverty / total_pop * 100) if total_pop > 0 else 0
    no_vehicle = sum(
        d.get("households_no_vehicle", 0)
        for d in census.data.values()
        if d.get("households_no_vehicle") is not None
    )

    area = iso["properties"]["area_sq_km"]
    county_ratio = area / avg_ks_county_area

    print(f"\n--- {name} ---")
    print(f"  Drive area:              {area:,.0f} sq km ({county_ratio:.1f}x avg KS county)")
    print(f"  Population:              {total_pop:,.0f}")
    print(f"  Avg. median income:      ${avg_income:,.0f}")
    print(f"  Poverty rate:            {poverty_rate:.1f}%")
    print(f"  Households w/o vehicle:  {no_vehicle:,}")
    print(f"  Service POIs (60-min):   {len(pois)}")

### What the Data Tells Us

1. **Counties are the wrong unit of analysis.** Each town's 60-minute driving community covers 10,000--13,000 sq km, spanning **5--6 average Kansas counties**. Dodge City's isochrone (~12,900 sq km) is the largest, shaped by its position at the intersection of US-50, US-56, and US-283. Reporting demographics for "Ford County" captures roughly one-sixth of the population that actually depends on Dodge City for services.

2. **Dodge City is the regional economic engine.** With the largest area, highest population (~89,000), highest average median income (~$76,000), and lowest poverty rate (~12%), Dodge City's meatpacking industry anchors southwestern Kansas. Cargill and National Beef draw workers from across the region, making the 60-minute community a genuine labor market -- not an abstraction.

3. **Liberal is the most isolated and most vulnerable.** It has the smallest driving area (~10,300 sq km), the highest poverty rate (~14%), and a critical healthcare gap: only 3 facilities within 4 minutes, then **nothing until 40 minutes**. If SouthWest Medical Center closes, Liberal's ~69,000-person community faces a 40-minute drive to the nearest hospital -- well beyond the golden hour for trauma care.

4. **Car ownership is universal, and that masks a deeper fragility.** Zero households without vehicles across all three communities means everyone currently has car access. But this also means the entire community model depends on private automobiles with no fallback. Rising fuel costs, aging drivers unable to renew licenses, vehicle breakdowns on a fixed income, and seasonal road closures all remove people from this system. There is no transit, no rideshare, no alternative.

5. **Walking access is negligible.** In Hays, a 15-minute walk covers ~3.4 sq km -- roughly **0.03%** of the 60-minute driving area. Walking provides access to about 24 services (downtown shops, a clinic, the library); driving provides 80. The walk-to-drive area ratio of ~3,200:1 means that a person without a car in rural Kansas has access to a fraction of a percent of the geographic community available to a driver.

6. **Services concentrate in town cores, creating systemic fragility.** Nearly all POI markers cluster within a few blocks of each town center. The remaining 99%+ of each isochrone's area -- thousands of square miles of wheat fields, rangeland, and small unincorporated settlements -- has zero services. This concentration is efficient as long as the hub town is healthy. But when a hub declines -- a hospital closes, a school consolidates, a major employer leaves -- the impact cascades through an area the size of several counties, with no nearby alternative.

---

## Limitations and Next Steps

### Data Limitations

- **OpenStreetMap completeness**: OSM coverage in rural western Kansas is less complete than in metro areas. Some businesses may be missing, and hours of operation are rarely tagged. The 80-POI cap on our searches means absolute counts are not directly comparable between towns -- the category *mix* is more informative than the total.

- **ACS margins of error**: Census data at the block group level in rural areas carries very large margins of error because block groups may contain only 600--1,000 people. The income and poverty estimates reported here should be treated as approximate, not precise.

- **Isochrone assumptions**: Driving isochrones assume posted speed limits and normal road conditions. In winter (ice storms are common in western Kansas) or during harvest season (slow farm equipment on two-lane highways), actual travel times can be significantly longer than modeled.

- **Block group size**: Rural block groups are physically enormous. A single block group may span hundreds of square miles, making choropleth maps less granular than in urban settings. The maps show *where people are* (clustered in town) but cannot show fine-grained variation within the vast rural block groups.

- **No-vehicle data**: The `households_no_vehicle` variable (Census B25044_003E) covers owner-occupied units only. Renter-occupied households without vehicles are tracked separately and are not included in our totals. The zero values reported here reflect near-universal car ownership among homeowners, but may undercount car-free renters.

### Suggested Extensions

- **More Kansas towns**: Add Garden City, Great Bend, Salina, or Colby to `analyze_multiple_pois` for a broader western Kansas comparison.
- **Broadband overlay**: Cross-reference FCC broadband coverage maps with isochrone areas to assess digital access alongside physical access.
- **School district analysis**: Use education POIs to assess whether school consolidation is placing unreasonable driving burdens on families.
- **Healthcare-specific deep dive**: Run `get_poi` with only `categories=["healthcare"]` for all three towns and compare the drive-time distributions to identify which communities are most vulnerable to closures.
- **Temporal analysis**: Compare ACS estimates across multiple years to track whether these communities are growing or declining.
- **Interactive maps**: Use `create_map(..., export_format='html')` for zoomable, clickable maps that county planners can explore in a browser.

---

## API Cheat Sheet

Quick reference for all SocialMapper functions used in this notebook:

| Function | Purpose | Key Parameters |
|---|---|---|
| `create_isochrone(location, travel_time, travel_mode)` | Travel-time polygon from a point | `travel_mode`: `"walk"`, `"drive"`, `"bike"` |
| `get_census_blocks(polygon=iso)` | Census block groups intersecting an area | Pass an isochrone or GeoJSON dict |
| `get_census_data(location, variables)` | ACS demographic data by block group | Variables: `"population"`, `"median_income"`, `"poverty"`, etc. |
| `create_map(data, column, ...)` | Choropleth map with optional overlays | `overlay_boundary`, `overlay_points`, `show_stats`, `cmap` |
| `get_poi(location, categories, ...)` | OpenStreetMap points of interest | `travel_time` triggers isochrone-bounded search |
| `analyze_multiple_pois(locations, ...)` | Multi-location demographic comparison | Returns rankings for each variable |
| `generate_report(data, format)` | Formatted HTML report from analysis data | `format`: `"html"` |
| `import_poi_csv(path, ...)` | Custom POI data from CSV file | Specify column mappings for lat/lon/name/type |

---

**Next notebook**: [09 -- When Walmart Closes: Grocery Access in Rural Kansas](09-walmart-closure-grocery-access.ipynb) -- a deep dive into grocery store dependency on Walmart in these same three towns, modeling what happens when the dominant grocery provider disappears.

*This notebook was created as part of the SocialMapper tutorial series. For more information, see the [SocialMapper documentation](https://github.com/mihiarc/socialmapper).*